# IWTC Tools: Incremental Rebuild

This notebook provides a fast, deterministic rebuild path for canonical index and graph artifacts for a single world repository.

It replaces full pipeline execution with targeted rebuilds based on simple rebuild flags or updated source files, writing all outputs directly to the canonical index directory, `indexes.path`.

### Design Reference
- indexing_system_overview.md

### Repository Descriptor
- world_repository.yml

### Notebook Phases
- P0: Parameters
- P1: Descriptor load, path resolution, and change detection
- P2.1: Load and normalize vocabulary tables
- P2.2: Vocabulary consistency checks
- P2.3: Build vocabulary lookup and normalized relationship semantics
- P3: Rebuild source-derived indexes
- P4: Rebuild evidence graphs
- P5: Rebuild structural semantic graphs

## Phase P0: Parameters

Define which world repository this notebook operates on and which index version it expects.

This notebook rebuilds index and graph data files. 

**IMPORTANT:** Unlike the bootstrapping notebooks, this notebook writes directly to the canonical index directory identified in the descriptor file under indexes.path.

In [1]:
# Phase P0: Parameters
LAST_PHASE_RUN = "P0"

# Absolute path to the world_repository.yml descriptor.
WORLD_REPOSITORY_DESCRIPTOR = (
    "/Users/charissophia/obsidian/Iron Wolf Trading Company/_meta/descriptors/world_repository.yml"
)

# Index version to load (must match previously generated artifacts)
INDEX_VERSION = "V0"
INDEX_VERSION_SUFFIX = INDEX_VERSION.lower()

# Optional: Force Rebuild
FORCE_REBUILD_SOURCES  = True
FORCE_REBUILD_EVIDENCE = True
FORCE_REBUILD_SEMANTIC = True

# Internal run metadata (do not edit)
from datetime import datetime
print(f"Notebook run initialized at: {datetime.now().strftime('%Y-%m-%d %H:%M')}")

print("\nForce rebuild:")
print(f"{'Raw sources:':16} {FORCE_REBUILD_SOURCES}")
print(f"{'Evidence graph:':16} {FORCE_REBUILD_EVIDENCE}")
print(f"{'Semantic graph:':16} {FORCE_REBUILD_SEMANTIC}")

Notebook run initialized at: 2026-03-27 16:22

Force rebuild:
Raw sources:     True
Evidence graph:  True
Semantic graph:  True


## Phase P1: Descriptor Load and Change Detection

Load the world repository descriptor, resolve all required paths, and detect whether source or vocabulary inputs have changed since the most recent graph build.

In [2]:
# Phase P1: Descriptor Load, Path Resolution, and Change Detection
LAST_PHASE_RUN = "P1"

from pathlib import Path
from datetime import datetime
import yaml
from IPython.display import display

p1_errors = []

# descriptor
descriptor_path = Path(WORLD_REPOSITORY_DESCRIPTOR)

if descriptor_path.exists():
    with descriptor_path.open("r", encoding="utf-8") as f:
        descriptor = yaml.safe_load(f)
    del f

    if not isinstance(descriptor, dict):
        raise ValueError("Descriptor file did not load as a dictionary.")
else:
    raise FileNotFoundError(f"Descriptor not found: {WORLD_REPOSITORY_DESCRIPTOR}")

del descriptor_path

# world root
WORLD_ROOT = descriptor.get("world_root")

if WORLD_ROOT and Path(WORLD_ROOT).is_absolute():
    WORLD_ROOT = Path(WORLD_ROOT)
else:
    raise ValueError("Descriptor must define world_root as an absolute path.")

# indexes
INDEXES_PATH = None
INDEXES_RELPATH = descriptor.get("indexes", {}).get("path")

if INDEXES_RELPATH:
    indexes_path = Path(INDEXES_RELPATH)

    if indexes_path.is_absolute():
        INDEXES_PATH = indexes_path
    else:
        INDEXES_PATH = WORLD_ROOT / indexes_path

    if not INDEXES_PATH.exists():
        p1_errors.append(f"Indexes path not found: {INDEXES_RELPATH}")

    del indexes_path
else:
    p1_errors.append("Descriptor missing indexes.path")

# vocab paths
VOCAB_ENTITIES_PATH = None
VOCAB_ENTITIES_RELPATH = None
VOCAB_ALIASES_PATH = None
VOCAB_ALIASES_RELPATH = None
VOCAB_RELATIONSHIPS_PATH = None
VOCAB_RELATIONSHIPS_RELPATH = None
VOCAB_PREDICATES_PATH = None
VOCAB_PREDICATES_RELPATH = None

def resolve_vocab_path(key):
    relpath = descriptor.get("vocabulary", {}).get(key)

    if relpath:
        path_obj = Path(relpath)

        if path_obj.is_absolute():
            abspath = path_obj
        else:
            abspath = WORLD_ROOT / path_obj

        if not abspath.exists():
            p1_errors.append(f"Vocabulary file not found: {relpath}")

        del path_obj
        return abspath, relpath

    p1_errors.append(f"Descriptor missing vocabulary.{key}")
    return None, None

VOCAB_ENTITIES_PATH, VOCAB_ENTITIES_RELPATH = resolve_vocab_path("entities")
VOCAB_ALIASES_PATH, VOCAB_ALIASES_RELPATH = resolve_vocab_path("aliases")
VOCAB_RELATIONSHIPS_PATH, VOCAB_RELATIONSHIPS_RELPATH = resolve_vocab_path("relationships")
VOCAB_PREDICATES_PATH, VOCAB_PREDICATES_RELPATH = resolve_vocab_path("predicates")

del resolve_vocab_path

# source paths
SOURCE_PATHS = []
read_paths = descriptor.get("sources", {}).get("read_paths", [])

if isinstance(read_paths, list) and len(read_paths) > 0:
    for entry in read_paths:
        if isinstance(entry, dict):
            relpath = entry.get("path")
            source_type = entry.get("type")
        else:
            relpath = entry
            source_type = None

        if relpath:
            path_obj = Path(relpath)

            if path_obj.is_absolute():
                abspath = path_obj
            else:
                abspath = WORLD_ROOT / path_obj

            SOURCE_PATHS.append(
                {
                    "path": abspath,
                    "relpath": relpath,
                    "type": source_type,
                }
            )

            if not abspath.exists():
                p1_errors.append(f"Source path not found: {relpath}")

            del path_obj, abspath
        else:
            p1_errors.append("A sources.read_paths entry is missing its path value.")

        del entry, relpath, source_type
else:
    p1_errors.append("Descriptor missing sources.read_paths or it is empty.")

if isinstance(read_paths, list) and len(read_paths) > 0 and len(SOURCE_PATHS) == 0:
    p1_errors.append("No valid source paths were resolved from sources.read_paths.")

del read_paths


# descriptor validation
if p1_errors:
    print("Phase P1 failed. Please verify the descriptor file.\n")
    for msg in p1_errors:
        print(f"- {msg}")
    raise ValueError("Phase P1 failed. See error list above.")

del p1_errors, descriptor, yaml


# latest graph timestamp
latest_graph_mtime = max(
    (p.stat().st_mtime for p in INDEXES_PATH.glob("graph_*.csv")),
    default=None
)

print(
    "Latest graph timestamp:",
    datetime.fromtimestamp(latest_graph_mtime)
    if latest_graph_mtime
    else "None found"
)


# changed inputs since most recent graph file
if latest_graph_mtime is None:
    CHANGED_SOURCES = True
    CHANGED_ENTITIES = True
    CHANGED_ALIASES = True
    CHANGED_PREDICATES = True
    CHANGED_RELATIONSHIPS = True
else:
    CHANGED_ENTITIES = VOCAB_ENTITIES_PATH.stat().st_mtime > latest_graph_mtime
    CHANGED_ALIASES = VOCAB_ALIASES_PATH.stat().st_mtime > latest_graph_mtime
    CHANGED_PREDICATES = VOCAB_PREDICATES_PATH.stat().st_mtime > latest_graph_mtime
    CHANGED_RELATIONSHIPS = VOCAB_RELATIONSHIPS_PATH.stat().st_mtime > latest_graph_mtime

    CHANGED_SOURCES = any(
        p.stat().st_mtime > latest_graph_mtime
        for source in SOURCE_PATHS
        for p in (
            source["path"].rglob("*")
            if source["path"].is_dir()
            else [source["path"]]
        )
        if p.is_file() and p.suffix.lower() in {".md", ".txt"}
    )

del latest_graph_mtime, datetime

# output
print("\nWorld Root:   ", WORLD_ROOT)
print("Indexes:      ", INDEXES_RELPATH)
print("Entities:     ", VOCAB_ENTITIES_RELPATH)
print("Aliases:      ", VOCAB_ALIASES_RELPATH)
print("Predicates:   ", VOCAB_PREDICATES_RELPATH)
print("Relationships:", VOCAB_RELATIONSHIPS_RELPATH)

print("\nSources:", len(SOURCE_PATHS), "read paths resolved")
display([s["relpath"] for s in SOURCE_PATHS])

print("\nPhase P1 OK: descriptor loaded, paths resolved, and changes detected.")

print("\nChanged inputs:")
print(f"{'Sources:':16} {CHANGED_SOURCES}")
print(f"{'Entities:':16} {CHANGED_ENTITIES}")
print(f"{'Aliases:':16} {CHANGED_ALIASES}")
print(f"{'Predicates:':16} {CHANGED_PREDICATES}")
print(f"{'Relationships:':16} {CHANGED_RELATIONSHIPS}")

Latest graph timestamp: 2026-03-27 16:15:10.171830

World Root:    /Users/charissophia/obsidian/Iron Wolf Trading Company
Indexes:       _meta/indexes
Entities:      _meta/indexes/vocab_entities.csv
Aliases:       _meta/indexes/vocab_aliases.csv
Predicates:    _meta/indexes/vocab_predicates.csv
Relationships: _meta/indexes/world_relationships.csv

Sources: 5 read paths resolved


['_local/auto_transcripts',
 '_local/pbp_transcripts',
 '_local/session_notes',
 '_local/planning_notes',
 '_local/recollections']


Phase P1 OK: descriptor loaded, paths resolved, and changes detected.

Changed inputs:
Sources:         False
Entities:        False
Aliases:         False
Predicates:      False
Relationships:   False


## Phase P2: Vocabulary Loads and Prep

Load the canonical vocabulary and semantic input tables, verify basic consistency across them, and prepare normalized lookup structures for later rebuild phases.

These files are small and may be reused across multiple later phases, so they are loaded once here and retained in memory.

Because these files are human-generated, column names are normalized here to a stable internal schema for later phases.

In [3]:
# Phase P2.1: Load and Normalize Canonical Vocabulary Tables
LAST_PHASE_RUN = "P2.1"

import pandas as pd

p2_errors = []


# Allowed column variants
ENTITY_COLS = {
    "entity_id": ["entity_id", "id"],
    "canonical": ["canonical", "canonical_name", "name"],
}

ALIAS_COLS = {
    "entity_id": ["entity_id", "id"],
    "alias": ["alias", "alt", "alternate"],
}

PREDICATE_COLS = {
    "predicate": ["predicate"],
    "reverse_predicate": ["reverse_predicate", "reverse"],
    "include": ["include"],
    "relationship_class": ["relationship_class", "class"],
    "priority": ["priority", "weight", "score"],
    "cost": ["cost"],
}

RELATIONSHIP_COLS = {
    "subject_id": ["subject_id", "subject", "from_id"],
    "predicate": ["predicate", "relationship", "relation", "edge", "verb"],
    "object_id": ["object_id", "object", "target_id", "to_id"],
}


def load_and_normalize_csv(path_obj, col_map, key_label):
    if path_obj.exists():
        raw_df = pd.read_csv(path_obj, dtype=str).fillna("")
    else:
        p2_errors.append(f"{key_label} file not found: {path_obj}")
        return None

    rename = {}
    for semantic, options in col_map.items():
        found = next((c for c in options if c in raw_df.columns), None)
        if found:
            rename[found] = semantic

    out = raw_df.rename(columns=rename)
    missing = [k for k in col_map.keys() if k not in out.columns]

    if len(missing) == 0:
        return out[list(col_map.keys())]

    p2_errors.append(f"[{key_label}] Missing required columns: {missing}")
    return None


# Load and normalize canonical vocab tables
DF_VOCAB_ENTITIES = load_and_normalize_csv(
    VOCAB_ENTITIES_PATH,
    ENTITY_COLS,
    "Entities",
)

DF_VOCAB_ALIASES = load_and_normalize_csv(
    VOCAB_ALIASES_PATH,
    ALIAS_COLS,
    "Aliases",
)

DF_VOCAB_PREDICATES = load_and_normalize_csv(
    VOCAB_PREDICATES_PATH,
    PREDICATE_COLS,
    "Predicates",
)

DF_VOCAB_RELATIONSHIPS = load_and_normalize_csv(
    VOCAB_RELATIONSHIPS_PATH,
    RELATIONSHIP_COLS,
    "Relationships",
)


# Normalize predicate data types
if DF_VOCAB_PREDICATES is not None:
    DF_VOCAB_PREDICATES["include"] = (
        DF_VOCAB_PREDICATES["include"]
        .str.strip()
        .str.lower()
        .map({"true": True, "false": False})
        .fillna(False)
    )

    DF_VOCAB_PREDICATES["priority"] = pd.to_numeric(
        DF_VOCAB_PREDICATES["priority"],
        errors="coerce",
    )

    DF_VOCAB_PREDICATES["cost"] = pd.to_numeric(
        DF_VOCAB_PREDICATES["cost"],
        errors="coerce",
    )

del ALIAS_COLS, ENTITY_COLS, PREDICATE_COLS, RELATIONSHIP_COLS

print("P2.1 checkpoint")

P2.1 checkpoint


In [4]:
# Phase P2.2: Vocabulary Consistency Checks
LAST_PHASE_RUN = "P2.2"

# ----------------------
# Entities internal consistency checks
# ----------------------
# missing entity_ids
if (DF_VOCAB_ENTITIES["entity_id"].fillna("").str.strip() == "").any():
    p2_errors.append(
        f"Entities has {(DF_VOCAB_ENTITIES['entity_id'].fillna('').str.strip() == '').sum()} row(s) with blank entity_id."
    )

# missing canonicals
if (DF_VOCAB_ENTITIES["canonical"].fillna("").str.strip() == "").any():
    p2_errors.append(
        f"Entities has {(DF_VOCAB_ENTITIES['canonical'].fillna('').str.strip() == '').sum()} row(s) with blank canonical."
    )

# duplicate entity_ids
dup_ids = (
    DF_VOCAB_ENTITIES["entity_id"]
    .fillna("")
    .str.strip()
    .pipe(lambda s: s[s.ne("")])
)
if (dup_ids.duplicated(keep=False)).any():
    p2_errors.append(
        f"Entities has duplicate entity_id values: {sorted(dup_ids[dup_ids.duplicated(keep=False)].unique())}"
    )
del dup_ids

# duplicate canonicals
dup_canonicals = (
    DF_VOCAB_ENTITIES["canonical"]
    .fillna("")
    .str.strip()
    .pipe(lambda s: s[s.ne("")])
)
if (dup_canonicals.duplicated(keep=False)).any():
    p2_errors.append(
        f"Entities has duplicate canonical values: {sorted(dup_canonicals[dup_canonicals.duplicated(keep=False)].unique())}"
    )
del dup_canonicals


# ----------------------
# Aliases internal consistency checks
# ----------------------
# missing entity_ids
if (DF_VOCAB_ALIASES["entity_id"].fillna("").str.strip() == "").any():
    p2_errors.append(
        f"Aliases has {(DF_VOCAB_ALIASES['entity_id'].fillna('').str.strip() == '').sum()} row(s) with blank entity_id."
    )

# missing aliases
if (DF_VOCAB_ALIASES["alias"].fillna("").str.strip() == "").any():
    p2_errors.append(
        f"Aliases has {(DF_VOCAB_ALIASES['alias'].fillna('').str.strip() == '').sum()} row(s) with blank alias."
    )

# duplicate aliases
dup_aliases = (
    DF_VOCAB_ALIASES["alias"]
    .fillna("")
    .str.strip()
    .pipe(lambda s: s[s.ne("")])
)
if (dup_aliases.duplicated(keep=False)).any():
    p2_errors.append(
        f"Aliases has duplicate alias values: {sorted(dup_aliases[dup_aliases.duplicated(keep=False)].unique())}"
    )
del dup_aliases


# ----------------------
# Alias cross checks
# ----------------------
# alias entity_ids must exist in entities
unknown_alias_entity_ids = (
    DF_VOCAB_ALIASES["entity_id"]
    .fillna("")
    .str.strip()
    .pipe(lambda s: s[s.ne("")])
    .pipe(lambda s: s[~s.isin(DF_VOCAB_ENTITIES["entity_id"].fillna("").str.strip())])
    .unique()
)
if len(unknown_alias_entity_ids) > 0:
    p2_errors.append(
        f"Aliases reference unknown entity_id values: {sorted(unknown_alias_entity_ids)}"
    )
del unknown_alias_entity_ids

# aliases must not duplicate canonicals
alias_duplicates_canonical = (
    DF_VOCAB_ALIASES["alias"]
    .fillna("")
    .str.strip()
    .pipe(lambda s: s[s.ne("")])
    .pipe(lambda s: s[s.isin(DF_VOCAB_ENTITIES["canonical"].fillna("").str.strip())])
    .unique()
)
if len(alias_duplicates_canonical) > 0:
    p2_errors.append(
        f"Aliases duplicates canonical values: {sorted(alias_duplicates_canonical)}"
    )
del alias_duplicates_canonical


# ----------------------
# Predicate internal consistency checks
# ----------------------
# blank predicates
if (DF_VOCAB_PREDICATES["predicate"].fillna("").str.strip() == "").any():
    p2_errors.append(
        f"Predicates has {(DF_VOCAB_PREDICATES['predicate'].fillna('').str.strip() == '').sum()} row(s) with blank predicate."
    )

# blank reverse_predicates
if (DF_VOCAB_PREDICATES["reverse_predicate"].fillna("").str.strip() == "").any():
    p2_errors.append(
        f"Predicates has {(DF_VOCAB_PREDICATES['reverse_predicate'].fillna('').str.strip() == '').sum()} row(s) with blank reverse_predicate."
    )

# blank relationship_class values
if (DF_VOCAB_PREDICATES["relationship_class"].fillna("").str.strip() == "").any():
    p2_errors.append(
        f"Predicates has {(DF_VOCAB_PREDICATES['relationship_class'].fillna('').str.strip() == '').sum()} row(s) with blank relationship_class."
    )

# duplicate predicates
dup_predicates = (
    DF_VOCAB_PREDICATES["predicate"]
    .fillna("")
    .str.strip()
    .pipe(lambda s: s[s.ne("")])
)
if (dup_predicates.duplicated(keep=False)).any():
    p2_errors.append(
        f"Predicates has duplicate predicate values: {sorted(dup_predicates[dup_predicates.duplicated(keep=False)].unique())}"
    )
del dup_predicates

# reverse_predicate must either equal predicate
# or not appear as any other predicate value
bad_reverse_links = (
    DF_VOCAB_PREDICATES.assign(
        predicate_clean=lambda df: df["predicate"].fillna("").str.strip(),
        reverse_clean=lambda df: df["reverse_predicate"].fillna("").str.strip(),
    )
    .loc[
        lambda df:
            (df["reverse_clean"] != "")
            & (df["reverse_clean"] != df["predicate_clean"])
            & (df["reverse_clean"].isin(df["predicate_clean"])),
        ["predicate_clean", "reverse_clean"]
    ]
    .drop_duplicates()
)
if len(bad_reverse_links) > 0:
    p2_errors.append(
        f"Predicates has reverse_predicate values that match other predicate rows: {bad_reverse_links.values.tolist()}"
    )
del bad_reverse_links


# ----------------------
# Relationship internal consistency checks
# ----------------------
# blank subject_ids
if (DF_VOCAB_RELATIONSHIPS["subject_id"].fillna("").str.strip() == "").any():
    p2_errors.append(
        f"Relationships has {(DF_VOCAB_RELATIONSHIPS['subject_id'].fillna('').str.strip() == '').sum()} row(s) with blank subject_id."
    )

# blank predicates
if (DF_VOCAB_RELATIONSHIPS["predicate"].fillna("").str.strip() == "").any():
    p2_errors.append(
        f"Relationships has {(DF_VOCAB_RELATIONSHIPS['predicate'].fillna('').str.strip() == '').sum()} row(s) with blank predicate."
    )

# blank object_ids
if (DF_VOCAB_RELATIONSHIPS["object_id"].fillna("").str.strip() == "").any():
    p2_errors.append(
        f"Relationships has {(DF_VOCAB_RELATIONSHIPS['object_id'].fillna('').str.strip() == '').sum()} row(s) with blank object_id."
    )

# duplicate relationship triples
dup_relationships = (
    DF_VOCAB_RELATIONSHIPS
    .assign(
        subject_id=lambda df: df["subject_id"].fillna("").str.strip(),
        predicate=lambda df: df["predicate"].fillna("").str.strip(),
        object_id=lambda df: df["object_id"].fillna("").str.strip(),
    )
    .query("subject_id != '' and predicate != '' and object_id != ''")
)

dup_relationship_rows = (
    dup_relationships.loc[
        dup_relationships.duplicated(subset=["subject_id", "predicate", "object_id"], keep=False),
        ["subject_id", "predicate", "object_id"]
    ]
    .drop_duplicates()
)

if len(dup_relationship_rows) > 0:
    p2_errors.append(
        "Relationships has duplicate (subject_id, predicate, object_id) rows: "
        f"{sorted(dup_relationship_rows.apply(tuple, axis=1).tolist())}"
    )

del dup_relationships, dup_relationship_rows


# ----------------------
# Relationship cross checks
# ----------------------
# relationship subject_id values must exist in entities
unknown_relationship_subject_ids = (
    DF_VOCAB_RELATIONSHIPS["subject_id"]
    .fillna("")
    .str.strip()
    .pipe(lambda s: s[s.ne("")])
    .pipe(lambda s: s[~s.isin(DF_VOCAB_ENTITIES["entity_id"].fillna("").str.strip())])
    .unique()
)
if len(unknown_relationship_subject_ids) > 0:
    p2_errors.append(
        f"Relationships reference unknown subject_id values: {sorted(unknown_relationship_subject_ids)}"
    )
del unknown_relationship_subject_ids

# relationship object_id values must exist in entities
unknown_relationship_object_ids = (
    DF_VOCAB_RELATIONSHIPS["object_id"]
    .fillna("")
    .str.strip()
    .pipe(lambda s: s[s.ne("")])
    .pipe(lambda s: s[~s.isin(DF_VOCAB_ENTITIES["entity_id"].fillna("").str.strip())])
    .unique()
)
if len(unknown_relationship_object_ids) > 0:
    p2_errors.append(
        f"Relationships reference unknown object_id values: {sorted(unknown_relationship_object_ids)}"
    )
del unknown_relationship_object_ids

# relationship predicate values must exist in predicates
unknown_relationship_predicates = (
    DF_VOCAB_RELATIONSHIPS["predicate"]
    .fillna("")
    .str.strip()
    .pipe(lambda s: s[s.ne("")])
    .pipe(lambda s: s[~s.isin(DF_VOCAB_PREDICATES["predicate"].fillna("").str.strip())])
    .unique()
)
if len(unknown_relationship_predicates) > 0:
    p2_errors.append(
        f"Relationships reference unknown predicate values: {sorted(unknown_relationship_predicates)}"
    )
del unknown_relationship_predicates

print("P2.2 checkpoint")

P2.2 checkpoint


In [5]:
# Phase P2.3: Build Vocabulary Lookup and Output Overall Results
LAST_PHASE_RUN = "P2.3"

import re

# ----------------------
# Build consolidated vocabulary lookup
# ----------------------
lookup_rows = []

for _, r in DF_VOCAB_ENTITIES.iterrows():
    if r.get("entity_id", "") and r.get("canonical", ""):
        lookup_rows.append(
            {
                "vocab": r["canonical"],
                "vocab_id": r["entity_id"],
                "canonical": r["canonical"],
                "vocab_kind": "canonical",
            }
        )
del _, r

canon_by_id = dict(zip(DF_VOCAB_ENTITIES["entity_id"], DF_VOCAB_ENTITIES["canonical"]))

for _, r in DF_VOCAB_ALIASES.iterrows():
    if r.get("entity_id", "") and r.get("alias", ""):
        lookup_rows.append(
            {
                "vocab": r["alias"],
                "vocab_id": r["entity_id"],
                "canonical": canon_by_id.get(r["entity_id"], ""),
                "vocab_kind": "alias",
            }
        )
del canon_by_id, _, r

DF_VOCAB_LOOKUP = (
    pd.DataFrame(lookup_rows)
    .drop_duplicates(subset=["vocab", "vocab_id"])
    .reset_index(drop=True)
)

DF_VOCAB_LOOKUP["vocab_len"] = DF_VOCAB_LOOKUP["vocab"].str.len()

DF_VOCAB_LOOKUP = (
    DF_VOCAB_LOOKUP
    .sort_values(["vocab_len", "vocab"], ascending=[False, True])
    .reset_index(drop=True)
)

DF_VOCAB_LOOKUP["vocab_norm"] = (
    DF_VOCAB_LOOKUP["vocab"]
    .astype(str)
    .str.strip()
    .str.lower()
)

patterns = []

for vocab in DF_VOCAB_LOOKUP["vocab"]:
    esc = re.escape(vocab)
    esc_ws = esc.replace(r"\ ", r"\s+")
    patterns.append(re.compile(rf"(?<!\w){esc_ws}(?!\w)", re.IGNORECASE))
    del esc, esc_ws

DF_VOCAB_LOOKUP["pattern"] = patterns
del vocab, patterns


# ----------------------
# Normalize semantic relationships
# ----------------------
DF_RELATIONSHIP_SEMANTICS = (
    DF_VOCAB_RELATIONSHIPS
    .assign(
        subject_type=lambda df: df["subject_id"].str.split("_", n=1).str[0],
        object_type=lambda df: df["object_id"].str.split("_", n=1).str[0],
    )
    .merge(
        DF_VOCAB_PREDICATES,
        on="predicate",
        how="left",
    )
    .assign(
        pair_type=lambda df: df["subject_type"] + "|" + df["object_type"],
        symmetric=lambda df: (
            df["predicate"].str.strip().str.lower()
            == df["reverse_predicate"].str.strip().str.lower()
        ),
    )
    [
        [
            "subject_id",
            "subject_type",
            "predicate",
            "object_id",
            "object_type",
            "pair_type",
            "include",
            "symmetric",
            "relationship_class",
        ]
    ]
    .sort_values(
        ["predicate", "subject_id", "object_id"],
        ascending=[True, True, True],
    )
    .reset_index(drop=True)
)


# ----------------------
# Output and cleanup
# ----------------------
if p2_errors:
    print("Phase P2 failed. Please verify the vocabulary files.\n")
    for msg in p2_errors:
        print(f"- {msg}")
    del msg
    raise ValueError("Phase P2 failed. See error list above.")

print(f"{'Entities:':14} {VOCAB_ENTITIES_RELPATH:40} ({len(DF_VOCAB_ENTITIES)} rows)")
print(f"{'Aliases:':14} {VOCAB_ALIASES_RELPATH:40} ({len(DF_VOCAB_ALIASES)} rows)")
print(f"{'Predicates:':14} {VOCAB_PREDICATES_RELPATH:40} ({len(DF_VOCAB_PREDICATES)} rows)")
print(f"{'Relationships:':14} {VOCAB_RELATIONSHIPS_RELPATH:40} ({len(DF_VOCAB_RELATIONSHIPS)} rows)")
print(f"{'Link vocabs:':14} {'(in memory)':40} ({len(DF_VOCAB_LOOKUP)} rows)")

print("\nNormalized semantic relationships:")
print(f"- Input relationships: {len(DF_VOCAB_RELATIONSHIPS):>8}")
print(f"- Output rows:         {len(DF_RELATIONSHIP_SEMANTICS):>8}")
print(f"- Distinct predicates: {DF_RELATIONSHIP_SEMANTICS['predicate'].nunique():>8}")
print(f"- Distinct pair types: {DF_RELATIONSHIP_SEMANTICS['pair_type'].nunique():>8}")
print()

display(DF_VOCAB_ENTITIES.head(3))
display(DF_VOCAB_ALIASES.head(3))
display(DF_VOCAB_PREDICATES.head(3))
display(DF_VOCAB_RELATIONSHIPS.head(3))
display(DF_VOCAB_LOOKUP.head(3))
display(DF_RELATIONSHIP_SEMANTICS.head(3))

print("\nPhase P2 OK: vocabulary tables loaded, validated, and prepared.")

del lookup_rows, p2_errors
del VOCAB_ENTITIES_PATH, VOCAB_ENTITIES_RELPATH
del VOCAB_ALIASES_PATH, VOCAB_ALIASES_RELPATH
del VOCAB_PREDICATES_PATH, VOCAB_PREDICATES_RELPATH
del VOCAB_RELATIONSHIPS_PATH, VOCAB_RELATIONSHIPS_RELPATH

Entities:      _meta/indexes/vocab_entities.csv         (402 rows)
Aliases:       _meta/indexes/vocab_aliases.csv          (83 rows)
Predicates:    _meta/indexes/vocab_predicates.csv       (33 rows)
Relationships: _meta/indexes/world_relationships.csv    (312 rows)
Link vocabs:   (in memory)                              (485 rows)

Normalized semantic relationships:
- Input relationships:      312
- Output rows:              312
- Distinct predicates:       19
- Distinct pair types:       23



,entity_id,canonical
0,art_folly,The Folly
1,art_glory,Unala's Glory
2,art_iris,IRIS


,entity_id,alias
0,art_folly,Folly
1,art_glory,Glory
2,pers_alivyre,Alivyre


,predicate,reverse_predicate,include,relationship_class,priority,cost
0,allied_with,allied_with,True,political,3,2
1,ambassador_to,has_ambassador,False,diplomatic,4,2
2,at_war_with,at_war_with,False,political,4,2


,subject_id,predicate,object_id
0,art_folly,belongs_to,org_folly
1,factn_burzath,current_member_of,org_orcs
2,factn_dakgorim,former_member_of,factn_ilguz


,vocab,vocab_id,canonical,vocab_kind,vocab_len,vocab_norm,pattern
0,Keeper of the Krach-ul Caves at Crafthold,off_lead_krach-ul_cavekeepers,Keeper of the Krach-ul Caves at Crafthold,canonical,41,keeper of the krach-ul caves at crafthold,re.compile('(?<!\\w)Keeper\\s+of\\s+the\\s+Kra...
1,Lead Handler of the Krach-ul at Crafthold,off_lead_krach-ul_handlers,Lead Handler of the Krach-ul at Crafthold,canonical,41,lead handler of the krach-ul at crafthold,re.compile('(?<!\\w)Lead\\s+Handler\\s+of\\s+t...
2,Krach-ul Animal Handlers at Crafthold,org_krach-ul_handlers,Krach-ul Animal Handlers at Crafthold,canonical,37,krach-ul animal handlers at crafthold,re.compile('(?<!\\w)Krach\\-ul\\s+Animal\\s+Ha...


,subject_id,subject_type,predicate,object_id,object_type,pair_type,include,symmetric,relationship_class
0,org_michaeline,org,allied_with,org_remedy,org,org|org,True,True,political
1,org_tolan_healers,org,allied_with,org_temple_airmid,org,org|org,True,True,political
2,pers_kavar,pers,allied_with,pers_henry,pers,pers|pers,True,True,political



Phase P2 OK: vocabulary tables loaded, validated, and prepared.


## Phase P3: Rebuild Source-Derived Indexes

Rebuild the source-derived entity index files if source content or source-matching vocabulary has changed.

Note that only .md and .txt files are currently loaded.

This phase writes directly to the canonical index directory.

In [6]:
# Phase P3: Rebuild Source-Derived Indexes
LAST_PHASE_RUN = "P3"

CHANGED_SOURCE_INDEXES = False

if (
    FORCE_REBUILD_SOURCES
    or CHANGED_SOURCES
    or CHANGED_ENTITIES
    or CHANGED_ALIASES
):
    print("Phase P3 will run:")
    print(f"{'  Force rebuild sources:':24} {FORCE_REBUILD_SOURCES}")
    print(f"{'  Changed sources:':24} {CHANGED_SOURCES}")
    print(f"{'  Changed entities:':24} {CHANGED_ENTITIES}")
    print(f"{'  Changed aliases:':24} {CHANGED_ALIASES}")

    # ----------------------
    # Discover source files
    # ----------------------
    SOURCE_FILES = []

    for source in SOURCE_PATHS:
        if source["path"].exists() and source["path"].is_dir():
            SOURCE_FILES.extend(
                {
                    "path": p,
                    "relpath": str(p.relative_to(WORLD_ROOT)),
                    "source_type": source["type"],
                }
                for p in source["path"].rglob("*")
                if p.is_file() and p.suffix.lower() in {".md", ".txt"}
            )

        elif source["path"].is_file() and source["path"].suffix.lower() in {".md", ".txt"}:
            SOURCE_FILES.append(
                {
                    "path": source["path"],
                    "relpath": str(source["path"].relative_to(WORLD_ROOT)),
                    "source_type": source["type"],
                }
            )

    del source

    print(f"\nSource files discovered: {len(SOURCE_FILES)}")
    display([s["relpath"] for s in SOURCE_FILES[:10]])
    if len(SOURCE_FILES) > 10:
        print("  ... (truncated)")

    print("\nBy source type:")
    print(
        "  "
        + pd.Series([s["source_type"] or "untyped" for s in SOURCE_FILES])
        .value_counts()
        .to_string()
        .replace("\n", "\n  ")
    )

    # ----------------------
    # Load source files into memory as raw lines
    # ----------------------
    LOADED_SOURCES = []

    for item in SOURCE_FILES:
        text = item["path"].read_text(encoding="utf-8", errors="replace")
        lines = text.splitlines()

        LOADED_SOURCES.append(
            {
                "path": item["path"],
                "relpath": item["relpath"],
                "source_type": item["source_type"],
                "lines": lines,
            }
        )

        del text, lines

    print(f"\nLoaded sources: {len(LOADED_SOURCES)}")
    for s in LOADED_SOURCES[:10]:
        print(f"  {s['relpath']}  ({len(s['lines'])} lines)")
    if len(LOADED_SOURCES) > 10:
        print("  ... (truncated)")

    if SOURCE_FILES:
        del item
    if LOADED_SOURCES:
        del s

    # ----------------------
    # Chunking v0 - consolidated header-driven chunking
    # ----------------------
    TIME_LIKE_REGEX = r"\d{1,2}:\d{2}(?::\d{2})?(?:\s*[AP]M)?"

    HEADER_REGEXES = [
        ("auto_ts", re.compile(r"^\s*\d{1,2}:\d{2}(?::\d{2})?\s*$")),
        ("pbp_hash", re.compile(rf"^\s*(?:\d+\.\s*)?(?:[*-]\s*)?###\s+.*{TIME_LIKE_REGEX}.*$")),
        ("pbp_forum", re.compile(rf"^\s*>?\s*\*\*.*{TIME_LIKE_REGEX}.*\*\*\s*$")),
        ("session", re.compile(r"^\s*(?:\d+\.\s*)?(?:[>#*_\-\s]+)?session\s+(?!notes\b)\S.*$", re.IGNORECASE)),
        ("md_heading", re.compile(r"^\s*(?:[*\-]\s*)?#{1,6}\s+\S.*$")),
    ]

    CHUNKS_V0 = []
    chunk_global_id = 1

    for src in LOADED_SOURCES:
        current_kind = "preamble"
        current_lines = []
        chunk_start_line = 1

        for idx, line in enumerate(src["lines"], start=1):
            matched_kind = next(
                (kind for kind, header_regex in HEADER_REGEXES if header_regex.match(line)),
                None,
            )

            if matched_kind:
                if current_lines:
                    CHUNKS_V0.append(
                        {
                            "chunk_id": chunk_global_id,
                            "source_type": src["source_type"],
                            "relpath": src["relpath"],
                            "start_line": chunk_start_line,
                            "end_line": idx - 1,
                            "header_kind": current_kind,
                            "lines": list(current_lines),
                        }
                    )
                    chunk_global_id += 1

                current_kind = matched_kind
                current_lines = [line]
                chunk_start_line = idx
            else:
                current_lines.append(line)

        if current_lines:
            CHUNKS_V0.append(
                {
                    "chunk_id": chunk_global_id,
                    "source_type": src["source_type"],
                    "relpath": src["relpath"],
                    "start_line": chunk_start_line,
                    "end_line": len(src["lines"]),
                    "header_kind": current_kind,
                    "lines": list(current_lines),
                }
            )
            chunk_global_id += 1

        if src["lines"]:
            del idx, line, matched_kind
        del current_kind, current_lines, chunk_start_line

    print(f"\nChunks created: {len(CHUNKS_V0)} from {len(LOADED_SOURCES)} files.")

    print("\nBy header kind:")
    print(
        "  "
        + pd.Series([c["header_kind"] for c in CHUNKS_V0])
        .value_counts()
        .to_string()
        .replace("\n", "\n  ")
    )

    display(
        pd.DataFrame(CHUNKS_V0)[
            ["chunk_id", "relpath", "start_line", "end_line", "header_kind"]
        ].head(10)
    )

    del TIME_LIKE_REGEX, HEADER_REGEXES, chunk_global_id
    if LOADED_SOURCES:
        del src

    # ----------------------
    # Link entities to chunks
    # ----------------------
    EXCLUDE_SOURCE_TYPES = {"auto_transcripts"}
    MAX_SNIPPET_CHARS = 220

    vocabs = DF_VOCAB_LOOKUP["vocab"].tolist()
    vocab_ids = DF_VOCAB_LOOKUP["vocab_id"].tolist()
    canonicals = DF_VOCAB_LOOKUP["canonical"].tolist()
    vocab_kinds = DF_VOCAB_LOOKUP["vocab_kind"].tolist()
    patterns = DF_VOCAB_LOOKUP["pattern"].tolist()

    rows = []
    mention_id = 1

    for chunk in CHUNKS_V0:
        source_type = chunk["source_type"]

        if source_type not in EXCLUDE_SOURCE_TYPES:
            lines = chunk["lines"]
            header_kind = chunk["header_kind"]

            if header_kind in {"pbp_hash", "pbp_forum", "session"} and lines:
                content_lines = lines[1:]
                content_start_line = (chunk["start_line"] or 1) + 1
            else:
                content_lines = lines
                content_start_line = chunk["start_line"] or 1

            concat_text = " ".join(" ".join(content_lines).split())

            if concat_text:
                if len(concat_text) > MAX_SNIPPET_CHARS:
                    snippet = concat_text[: MAX_SNIPPET_CHARS - 3] + "..."
                else:
                    snippet = concat_text

                consumed = [False] * len(concat_text)

                for vocab, vocab_id, canonical, vocab_kind, pat in zip(
                    vocabs, vocab_ids, canonicals, vocab_kinds, patterns
                ):
                    hits = [(m.start(), m.end()) for m in pat.finditer(concat_text)]

                    if hits:
                        kept = []

                        for a, b in hits:
                            if a >= 0 and b > a and not any(consumed[a:b]):
                                kept.append((a, b))
                        del a, b

                        if kept:
                            for a, b in kept:
                                for k in range(a, b):
                                    consumed[k] = True

                            rows.append(
                                {
                                    "mention_id": mention_id,
                                    "entity_id": vocab_id,
                                    "canonical": canonical,
                                    "matched_vocab": vocab,
                                    "match_kind": vocab_kind,
                                    "match_count_in_chunk": len(kept),
                                    "chunk_id": chunk["chunk_id"],
                                    "source_id": chunk["relpath"],
                                    "source_type": chunk["source_type"],
                                    "relpath": chunk["relpath"],
                                    "chunk_start_line": chunk["start_line"],
                                    "chunk_end_line": chunk["end_line"],
                                    "header_kind": chunk["header_kind"],
                                    "content_start_line": content_start_line,
                                    "snippet": snippet,
                                }
                            )
                            mention_id += 1

                            del a, b, k
                        del kept
                    del hits

                del vocab, vocab_id, canonical, vocab_kind, pat
                del consumed, snippet

            del content_lines, content_start_line, concat_text
            del lines, header_kind

        del source_type

    ENTITY_MENTIONS_V0 = pd.DataFrame(rows)

    print(
        f"\nChunks scanned for entity linking: "
        f"{sum(1 for c in CHUNKS_V0 if c['source_type'] not in EXCLUDE_SOURCE_TYPES)}"
    )
    print(f"Entity mentions: {len(ENTITY_MENTIONS_V0)} rows")

    if len(ENTITY_MENTIONS_V0) > 0:
        display(ENTITY_MENTIONS_V0.head(10))

    del EXCLUDE_SOURCE_TYPES, MAX_SNIPPET_CHARS
    del vocabs, vocab_ids, canonicals, vocab_kinds, patterns
    del rows, mention_id
    if CHUNKS_V0:
        del chunk

    # ----------------------
    # Build source-derived indexes
    # ----------------------
    INDEX_SOURCE_FILES_V0 = (
        pd.DataFrame(
            {
                "source_id": [s["relpath"] for s in LOADED_SOURCES],
                "relpath": [s["relpath"] for s in LOADED_SOURCES],
                "source_type": [s["source_type"] for s in LOADED_SOURCES],
            }
        )
        .drop_duplicates()
        .sort_values(["source_type", "relpath"], ascending=[True, True])
        .reset_index(drop=True)
    )

    INDEX_CHUNK_TO_ENTITIES_V0 = (
        ENTITY_MENTIONS_V0.groupby(
            ["chunk_id", "source_id", "source_type", "relpath", "chunk_start_line", "chunk_end_line"],
            as_index=False
        )
        .agg(
            entity_ids=("entity_id", lambda s: "|".join(pd.unique(s.astype(str)))),
            canonicals=("canonical", lambda s: "|".join(pd.unique(s.astype(str)))),
            entity_count=("entity_id", pd.Series.nunique),
            matched_vocabs=("matched_vocab", lambda s: "|".join(pd.unique(s.astype(str)))),
            match_kinds=("match_kind", lambda s: "|".join(pd.unique(s.astype(str)))),
        )
        .sort_values(["relpath", "chunk_start_line", "chunk_id"], ascending=[True, True, True])
        .reset_index(drop=True)
    )

    INDEX_ENTITY_TO_CHUNKS_V0 = (
        ENTITY_MENTIONS_V0.groupby(["entity_id", "canonical"], as_index=False)
        .agg(
            chunk_ids=("chunk_id", lambda s: "|".join(str(v) for v in sorted(pd.unique(s)))),
            chunk_count=("chunk_id", pd.Series.nunique),
            file_relpaths=("relpath", lambda s: "|".join(sorted(pd.unique(s.astype(str))))),
            file_count=("relpath", pd.Series.nunique),
        )
        .sort_values(["canonical", "entity_id"], ascending=[True, True])
        .reset_index(drop=True)
    )

    # ----------------------
    # Write outputs
    # ----------------------
    INDEX_SOURCE_FILES_V0.to_csv(
        INDEXES_PATH / f"index_source_files_{INDEX_VERSION_SUFFIX}.csv",
        index=False,
        encoding="utf-8",
    )

    INDEX_CHUNK_TO_ENTITIES_V0.to_csv(
        INDEXES_PATH / f"index_chunk_to_entities_{INDEX_VERSION_SUFFIX}.csv",
        index=False,
        encoding="utf-8",
    )

    INDEX_ENTITY_TO_CHUNKS_V0.to_csv(
        INDEXES_PATH / f"index_entity_to_chunks_{INDEX_VERSION_SUFFIX}.csv",
        index=False,
        encoding="utf-8",
    )

    print("\nWrote:")
    print(f"  {INDEXES_RELPATH}/index_source_files_{INDEX_VERSION_SUFFIX}.csv")
    print(f"  {INDEXES_RELPATH}/index_chunk_to_entities_{INDEX_VERSION_SUFFIX}.csv")
    print(f"  {INDEXES_RELPATH}/index_entity_to_chunks_{INDEX_VERSION_SUFFIX}.csv")

    CHANGED_SOURCE_INDEXES = True

    # ----------------------
    # Cleanup
    # ----------------------
    del LOADED_SOURCES, CHUNKS_V0, ENTITY_MENTIONS_V0, SOURCE_FILES

else:
    print("Phase P3 skipped.")
    print("No source-derived inputs changed.")

print("\nChanged source indexes:", CHANGED_SOURCE_INDEXES)

Phase P3 will run:
  Force rebuild sources: True
  Changed sources:       False
  Changed entities:      False
  Changed aliases:       False

Source files discovered: 130


['_local/auto_transcripts/iwtc session 103.txt',
 '_local/auto_transcripts/iwtc session 088.txt',
 '_local/auto_transcripts/iwtc session 077.txt',
 '_local/auto_transcripts/iwtc session 063.txt',
 '_local/auto_transcripts/iwtc session 062.txt',
 '_local/auto_transcripts/iwtc session 089.txt',
 '_local/auto_transcripts/iwtc session 102.txt',
 '_local/auto_transcripts/iwtc session 048.txt',
 '_local/auto_transcripts/iwtc session 060.txt',
 '_local/auto_transcripts/iwtc session 074.txt']

  ... (truncated)

By source type:
  auto_transcripts    105
  planning_notes       11
  pbp_transcripts       9
  session_notes         5

Loaded sources: 130
  _local/auto_transcripts/iwtc session 103.txt  (3425 lines)
  _local/auto_transcripts/iwtc session 088.txt  (3135 lines)
  _local/auto_transcripts/iwtc session 077.txt  (3515 lines)
  _local/auto_transcripts/iwtc session 063.txt  (3597 lines)
  _local/auto_transcripts/iwtc session 062.txt  (3683 lines)
  _local/auto_transcripts/iwtc session 089.txt  (3683 lines)
  _local/auto_transcripts/iwtc session 102.txt  (2555 lines)
  _local/auto_transcripts/iwtc session 048.txt  (4061 lines)
  _local/auto_transcripts/iwtc session 060.txt  (3193 lines)
  _local/auto_transcripts/iwtc session 074.txt  (4013 lines)
  ... (truncated)

Chunks created: 170131 from 130 files.

By header kind:
  auto_ts       168537
  pbp_hash         860
  md_heading       439
  session          179
  preamble         116


,chunk_id,relpath,start_line,end_line,header_kind
0,1,_local/auto_transcripts/iwtc session 103.txt,1,1,preamble
1,2,_local/auto_transcripts/iwtc session 103.txt,2,3,auto_ts
2,3,_local/auto_transcripts/iwtc session 103.txt,4,5,auto_ts
3,4,_local/auto_transcripts/iwtc session 103.txt,6,7,auto_ts
4,5,_local/auto_transcripts/iwtc session 103.txt,8,9,auto_ts
5,6,_local/auto_transcripts/iwtc session 103.txt,10,11,auto_ts
6,7,_local/auto_transcripts/iwtc session 103.txt,12,13,auto_ts
7,8,_local/auto_transcripts/iwtc session 103.txt,14,15,auto_ts
8,9,_local/auto_transcripts/iwtc session 103.txt,16,17,auto_ts
9,10,_local/auto_transcripts/iwtc session 103.txt,18,19,auto_ts



Chunks scanned for entity linking: 1486
Entity mentions: 5369 rows


,mention_id,entity_id,canonical,matched_vocab,match_kind,match_count_in_chunk,chunk_id,source_id,source_type,relpath,chunk_start_line,chunk_end_line,header_kind,content_start_line,snippet
0,1,play_bysickle,Bysickle,Bysickle,canonical,1,168647,_local/pbp_transcripts/PbP14 - Recon.md,pbp_transcripts,_local/pbp_transcripts/PbP14 - Recon.md,3,8,pbp_hash,4,* @Bysickle **Henry** slips between shadow and...
1,2,pers_ronric,Ronric,Ronric,canonical,5,168647,_local/pbp_transcripts/PbP14 - Recon.md,pbp_transcripts,_local/pbp_transcripts/PbP14 - Recon.md,3,8,pbp_hash,4,* @Bysickle **Henry** slips between shadow and...
2,3,pers_victor,Victor D Evernight,Victor,alias,5,168647,_local/pbp_transcripts/PbP14 - Recon.md,pbp_transcripts,_local/pbp_transcripts/PbP14 - Recon.md,3,8,pbp_hash,4,* @Bysickle **Henry** slips between shadow and...
3,4,pers_henry,Henry Sleepsong,Henry,alias,1,168647,_local/pbp_transcripts/PbP14 - Recon.md,pbp_transcripts,_local/pbp_transcripts/PbP14 - Recon.md,3,8,pbp_hash,4,* @Bysickle **Henry** slips between shadow and...
4,5,pers_light,Light of Ironfang,Light,alias,1,168647,_local/pbp_transcripts/PbP14 - Recon.md,pbp_transcripts,_local/pbp_transcripts/PbP14 - Recon.md,3,8,pbp_hash,4,* @Bysickle **Henry** slips between shadow and...
5,6,pers_gina,Gina,Gina,canonical,4,168647,_local/pbp_transcripts/PbP14 - Recon.md,pbp_transcripts,_local/pbp_transcripts/PbP14 - Recon.md,3,8,pbp_hash,4,* @Bysickle **Henry** slips between shadow and...
6,7,play_crowe,Crowe,CroweTheDualityKing,alias,1,168648,_local/pbp_transcripts/PbP14 - Recon.md,pbp_transcripts,_local/pbp_transcripts/PbP14 - Recon.md,9,16,pbp_hash,10,* @Kalina Hitana @CroweTheDualityKing Alivyre ...
7,8,play_kalina,Kalina,Kalina Hitana,alias,1,168648,_local/pbp_transcripts/PbP14 - Recon.md,pbp_transcripts,_local/pbp_transcripts/PbP14 - Recon.md,9,16,pbp_hash,10,* @Kalina Hitana @CroweTheDualityKing Alivyre ...
8,9,creat_little_bear,Little Bear,Little Bear,canonical,1,168648,_local/pbp_transcripts/PbP14 - Recon.md,pbp_transcripts,_local/pbp_transcripts/PbP14 - Recon.md,9,16,pbp_hash,10,* @Kalina Hitana @CroweTheDualityKing Alivyre ...
9,10,org_academy_elysia,Elysian Academy,Academy,alias,1,168648,_local/pbp_transcripts/PbP14 - Recon.md,pbp_transcripts,_local/pbp_transcripts/PbP14 - Recon.md,9,16,pbp_hash,10,* @Kalina Hitana @CroweTheDualityKing Alivyre ...



Wrote:
  _meta/indexes/index_source_files_v0.csv
  _meta/indexes/index_chunk_to_entities_v0.csv
  _meta/indexes/index_entity_to_chunks_v0.csv

Changed source indexes: True


## Phase P4: Rebuild Evidence Graphs

Rebuild canonical evidence graph nodes and edges from the source-derived indexes and normalized vocabulary lookup.

In [9]:
# Phase P4: Rebuild Evidence Graphs
LAST_PHASE_RUN = "P4"
CHANGED_EVIDENCE_GRAPHS = False

if (
    FORCE_REBUILD_EVIDENCE
    or CHANGED_SOURCE_INDEXES
    or CHANGED_ENTITIES
    or CHANGED_ALIASES
):
    print("Phase P4 will run:")
    print(f"{'  Force rebuild evidence:':24} {FORCE_REBUILD_EVIDENCE}")
    print(f"{'  Changed indexes:':24} {CHANGED_SOURCE_INDEXES}")
    print(f"{'  Changed entities:':24} {CHANGED_ENTITIES}")
    print(f"{'  Changed aliases:':24} {CHANGED_ALIASES}")

    # ----------------------
    # Ensure required source indexes are available
    # ----------------------
    if "INDEX_SOURCE_FILES_V0" not in globals():
        INDEX_SOURCE_FILES_V0 = pd.read_csv(
            INDEXES_PATH / f"index_source_files_{INDEX_VERSION_SUFFIX}.csv"
        )
    
    if "INDEX_CHUNK_TO_ENTITIES_V0" not in globals():
        INDEX_CHUNK_TO_ENTITIES_V0 = pd.read_csv(
            INDEXES_PATH / f"index_chunk_to_entities_{INDEX_VERSION_SUFFIX}.csv"
        )

    
    # ----------------------
    # Build nodes
    # ----------------------
    nodes = []

    # Entity nodes
    nodes.append(
        DF_VOCAB_ENTITIES.assign(
            node_id=lambda d: d["entity_id"].astype(str),
            node_type=lambda d: d["entity_id"].astype(str).str.split("_", n=1).str[0],
            label=lambda d: d["canonical"].astype(str),
        ).loc[:, ["node_id", "node_type", "label"]]
    )

    # Vocab nodes
    nodes.append(
        DF_VOCAB_LOOKUP.assign(
            node_id=lambda d: "vocab:" + d["vocab_norm"].astype(str),
            node_type="vocab",
            label=lambda d: d["vocab"].astype(str),
        ).loc[:, ["node_id", "node_type", "label"]]
    )

    # Chunk nodes
    nodes.append(
        INDEX_CHUNK_TO_ENTITIES_V0.assign(
            node_id=lambda d: "chunk_" + d["chunk_id"].astype(int).astype(str),
            node_type="chunk",
            label=lambda d: "chunk_" + d["chunk_id"].astype(int).astype(str),
        ).loc[:, ["node_id", "node_type", "label"]]
    )

    # File nodes
    nodes.append(
        INDEX_SOURCE_FILES_V0.assign(
            node_id=lambda d: "file:" + d["relpath"].astype(str),
            node_type=lambda d: d["source_type"].astype(str),
            label=lambda d: d["relpath"].astype(str),
        ).loc[:, ["node_id", "node_type", "label"]]
    )

    GRAPH_EVIDENCE_NODES_V0 = (
        pd.concat(nodes, ignore_index=True)
        .drop_duplicates(subset=["node_id"])
        .sort_values(["node_type", "node_id"])
        .reset_index(drop=True)
    )
    del nodes

    print(f"\nEvidence nodes built: {len(GRAPH_EVIDENCE_NODES_V0)}")

    print("\nCounts by node_type:")
    display(GRAPH_EVIDENCE_NODES_V0["node_type"].value_counts().to_frame("count"))

    print("\nSample evidence nodes:")
    display(GRAPH_EVIDENCE_NODES_V0.sample(min(5, len(GRAPH_EVIDENCE_NODES_V0)), random_state=7))

    # ----------------------
    # Build edges
    # ----------------------
    from itertools import combinations

    edge_rows = []

    # contains, mentions, cooccurs_with
    for r in INDEX_CHUNK_TO_ENTITIES_V0.loc[
        :, ["chunk_id", "relpath", "matched_vocabs", "entity_ids"]
    ].itertuples(index=False):
        chunk_node = f"chunk_{int(r.chunk_id)}"
        file_node = f"file:{r.relpath}"

        # file contains chunk
        edge_rows.append((file_node, "contains", chunk_node, pd.NA))

        # chunk mentions vocab
        norm_vocabs = [
            v.strip().lower()
            for v in str(r.matched_vocabs).split("|")
            if v.strip()
        ]
        edge_rows.extend(
            (chunk_node, "mentions", f"vocab:{norm_vocab}", pd.NA)
            for norm_vocab in norm_vocabs
        )

        # entity cooccurrence within chunk
        entity_ids = sorted(
            {
                e.strip()
                for e in str(r.entity_ids).split("|")
                if e.strip()
            }
        )
        edge_rows.extend(
            (left_id, "cooccurs_with", right_id, 1)
            for left_id, right_id in combinations(entity_ids, 2)
        )

        del chunk_node, file_node, norm_vocabs, entity_ids, r

    # refers_to
    for r in DF_VOCAB_LOOKUP.loc[:, ["vocab_norm", "vocab_id"]].itertuples(index=False):
        subject = f"vocab:{str(r.vocab_norm).strip()}"
        object_ = str(r.vocab_id).strip()

        edge_rows.append((subject, "refers_to", object_, pd.NA))
        del subject, object_, r

    GRAPH_EVIDENCE_EDGES_V0 = (
        pd.DataFrame(edge_rows, columns=["subject", "predicate", "object", "weight"])
        .groupby(["subject", "predicate", "object"], as_index=False)
        .agg(weight=("weight", lambda s: s.sum(min_count=1)))
        .sort_values(["predicate", "subject", "object"], ascending=[True, True, True])
        .reset_index(drop=True)
    )

    del edge_rows, combinations

    print(f"\nEvidence edges built: {len(GRAPH_EVIDENCE_EDGES_V0)}")

    print("\nEvidence edge counts by predicate:")
    display(
        GRAPH_EVIDENCE_EDGES_V0
        .groupby("predicate", as_index=False)
        .size()
        .rename(columns={"size": "edge_count"})
        .sort_values("edge_count", ascending=False)
        .reset_index(drop=True)
    )

    print("\nSample evidence edges:")
    display(GRAPH_EVIDENCE_EDGES_V0.sample(min(5, len(GRAPH_EVIDENCE_EDGES_V0)), random_state=7))

    # ----------------------
    # Write indexes out to files
    # ----------------------
    GRAPH_EVIDENCE_NODES_V0.to_csv(
        INDEXES_PATH / f"graph_evidence_nodes_{INDEX_VERSION_SUFFIX}.csv",
        index=False,
        encoding="utf-8",
    )
    GRAPH_EVIDENCE_EDGES_V0.to_csv(
        INDEXES_PATH / f"graph_evidence_edges_{INDEX_VERSION_SUFFIX}.csv",
        index=False,
        encoding="utf-8",
    )

    print("\nWrote:")
    print(f"  {INDEXES_RELPATH}/graph_evidence_nodes_{INDEX_VERSION_SUFFIX}.csv")
    print(f"  {INDEXES_RELPATH}/graph_evidence_edges_{INDEX_VERSION_SUFFIX}.csv")

    display(GRAPH_EVIDENCE_NODES_V0.head(3))
    display(GRAPH_EVIDENCE_EDGES_V0.head(3))

    CHANGED_EVIDENCE_GRAPHS = True

else:
    print("Phase P4 skipped.")
    print("No inputs changed.")

print("\nChanged evidence graphs:", CHANGED_EVIDENCE_GRAPHS)

Phase P4 will run:
  Force rebuild evidence: True
  Changed indexes:       True
  Changed entities:      False
  Changed aliases:       False

Evidence nodes built: 2161

Counts by node_type:


,count
node_type,
chunk,1144
vocab,485
pers,184
auto_transcripts,105
off,64
org,53
loc,34
role,31
play,16



Sample evidence nodes:


,node_id,node_type,label
1818,vocab:eira sleepsong,vocab,Eira Sleepsong
262,chunk_168834,chunk,chunk_168834
2138,vocab:victor d evernight,vocab,Victor D Evernight
1771,vocab:crafthold reeve,vocab,Crafthold Reeve
60,file:_local/auto_transcripts/iwtc session 062.txt,auto_transcripts,_local/auto_transcripts/iwtc session 062.txt



Evidence edges built: 15061

Evidence edge counts by predicate:


...

,subject,predicate,object,weight
11374,chunk_169530,mentions,vocab:crafthold,NaN
3528,loc_jotunheim,cooccurs_with,pers_alivyre,1.0
10877,chunk_169370,mentions,vocab:faeryne,NaN
9275,chunk_168659,mentions,vocab:piers,NaN
9004,pers_soren,cooccurs_with,pers_urgulk,3.0



Wrote:
  _meta/indexes/graph_evidence_nodes_v0.csv
  _meta/indexes/graph_evidence_edges_v0.csv


,node_id,node_type,label
0,art_folly,art,The Folly
1,art_glory,art,Unala's Glory
2,art_iris,art,IRIS


,subject,predicate,object,weight
0,file:_local/pbp_transcripts/PbP10 - The Second...,contains,chunk_169186,NaN
1,file:_local/pbp_transcripts/PbP10 - The Second...,contains,chunk_169187,NaN
2,file:_local/pbp_transcripts/PbP10 - The Second...,contains,chunk_169188,NaN



Changed evidence graphs: True


## Phase P5: Rebuild Structural Semantic Graphs

Rebuild canonical structural semantic graph nodes and edges from normalized relationship semantics.

In [8]:
# Phase P5: Rebuild Structural Semantic Graphs
LAST_PHASE_RUN = "P5"
CHANGED_SEMANTIC_GRAPHS = False

if (
    FORCE_REBUILD_SEMANTIC
    or CHANGED_ENTITIES
    or CHANGED_ALIASES
    or CHANGED_PREDICATES
    or CHANGED_RELATIONSHIPS
):
    print("Phase P5 will run:")
    print(f"{'  Force rebuild semantic:':26} {FORCE_REBUILD_SEMANTIC}")
    print(f"{'  Changed entities:':26} {CHANGED_ENTITIES}")
    print(f"{'  Changed aliases:':26} {CHANGED_ALIASES}")
    print(f"{'  Changed predicates:':26} {CHANGED_PREDICATES}")
    print(f"{'  Changed relationships:':26} {CHANGED_RELATIONSHIPS}")

    # ----------------------
    # Build nodes
    # ----------------------
    GRAPH_SEMANTIC_NODES_V0 = (
        pd.concat(
            [
                DF_RELATIONSHIP_SEMANTICS.loc[lambda df: df["include"], "subject_id"],
                DF_RELATIONSHIP_SEMANTICS.loc[lambda df: df["include"], "object_id"],
            ],
            ignore_index=True,
        )
        .drop_duplicates()
        .to_frame(name="node_id")
        .assign(
            node_type=lambda df: df["node_id"].str.split("_", n=1).str[0]
        )
        .merge(
            DF_VOCAB_ENTITIES[["entity_id", "canonical"]],
            left_on="node_id",
            right_on="entity_id",
            how="left",
        )
        .drop(columns="entity_id")
        .rename(columns={"canonical": "label"})
        .sort_values(["node_type", "node_id"], ascending=[True, True])
        .reset_index(drop=True)
    )

    print(f"\nSemantic nodes built: {len(GRAPH_SEMANTIC_NODES_V0)}")

    print("\nCounts by node_type:")
    display(GRAPH_SEMANTIC_NODES_V0["node_type"].value_counts().to_frame("count"))

    print("\nSample semantic nodes:")
    display(GRAPH_SEMANTIC_NODES_V0.sample(min(5, len(GRAPH_SEMANTIC_NODES_V0)), random_state=7))

    # ----------------------
    # Build edges
    # ----------------------
    GRAPH_SEMANTIC_EDGES_V0 = (
        DF_RELATIONSHIP_SEMANTICS
        .loc[lambda df: df["include"]]
        [
            [
                "subject_id",
                "predicate",
                "object_id",
                "subject_type",
                "object_type",
                "pair_type",
                "symmetric",
                "relationship_class",
            ]
        ]
        .sort_values(
            ["predicate", "subject_id", "object_id"],
            ascending=[True, True, True],
        )
        .reset_index(drop=True)
    )

    print(f"\nSemantic edges built: {len(GRAPH_SEMANTIC_EDGES_V0)}")

    print("\nSemantic edge counts by predicate:")
    display(
        GRAPH_SEMANTIC_EDGES_V0
        .groupby("predicate", as_index=False)
        .size()
        .rename(columns={"size": "edge_count"})
        .sort_values("edge_count", ascending=False)
        .reset_index(drop=True)
    )

    print("\nSample semantic edges:")
    display(GRAPH_SEMANTIC_EDGES_V0.sample(min(5, len(GRAPH_SEMANTIC_EDGES_V0)), random_state=7))

    # ----------------------
    # Write indexes out to files
    # ----------------------
    GRAPH_SEMANTIC_NODES_V0.to_csv(
        INDEXES_PATH / f"graph_semantic_nodes_{INDEX_VERSION_SUFFIX}.csv",
        index=False,
        encoding="utf-8",
    )
    GRAPH_SEMANTIC_EDGES_V0.to_csv(
        INDEXES_PATH / f"graph_semantic_edges_{INDEX_VERSION_SUFFIX}.csv",
        index=False,
        encoding="utf-8",
    )

    print("\nWrote:")
    print(f"  {INDEXES_RELPATH}/graph_semantic_nodes_{INDEX_VERSION_SUFFIX}.csv")
    print(f"  {INDEXES_RELPATH}/graph_semantic_edges_{INDEX_VERSION_SUFFIX}.csv")

    display(GRAPH_SEMANTIC_NODES_V0.head(3))
    display(GRAPH_SEMANTIC_EDGES_V0.head(3))

    CHANGED_SEMANTIC_GRAPHS = True

else:
    print("Phase P5 skipped.")
    print("No inputs changed.")

print("\nChanged semantic graphs:", CHANGED_SEMANTIC_GRAPHS)

Phase P5 will run:
  Force rebuild semantic:  True
  Changed entities:        False
  Changed aliases:         False
  Changed predicates:      False
  Changed relationships:   False

Semantic nodes built: 302

Counts by node_type:


,count
node_type,
pers,148
off,62
org,50
role,25
loc,10
factn,7



Sample semantic nodes:


,node_id,node_type,label
223,pers_mowser,pers,Mowser
57,off_lead_river_valley_farms,off,River Valley Farm Warden
263,pers_unala,pers,Unala Vortoise
66,off_lead_wolfstream,off,Abbess of Wolfstream Abbey
114,org_rarmal,org,Rarmal Garrison



Semantic edges built: 308

Semantic edge counts by predicate:


...

,subject_id,predicate,object_id,subject_type,object_type,pair_type,symmetric,relationship_class
37,org_krach-ul_cavekeepers,current_member_of,org_krach-ul,org,org,org|org,False,membership
170,pers_harold,former_holder_of,off_ops_wolfstream_herbalist,pers,off,pers|off,False,organizational
256,off_lead_jackals,leader_of,org_jackals,off,org,off|org,False,leadership
60,pers_charlie,current_member_of,role_ops_folly_cabin_boy,pers,role,pers|role,False,membership
153,role_logis_folly_quartermaster,current_member_of,org_folly,role,org,role|org,False,membership



Wrote:
  _meta/indexes/graph_semantic_nodes_v0.csv
  _meta/indexes/graph_semantic_edges_v0.csv


,node_id,node_type,label
0,factn_burzath,factn,Burzath Clan
1,factn_dakgorim,factn,Dakgorim Warband
2,factn_huvorgha,factn,Huvorgha Clan


,subject_id,predicate,object_id,subject_type,object_type,pair_type,symmetric,relationship_class
0,org_michaeline,allied_with,org_remedy,org,org,org|org,True,political
1,org_tolan_healers,allied_with,org_temple_airmid,org,org,org|org,True,political
2,pers_kavar,allied_with,pers_henry,pers,pers,pers|pers,True,political



Changed semantic graphs: True
